# Code 8.1.2: A bigram baseline on tiny Shakespeare

Tokenizes tiny Shakespeare with GPT-2's tokenizer, holds out the last 10% for validation, and scores add-$\alpha$ bigram models against the uniform model.


In [ ]:
import math, os, urllib.request
from collections import Counter
import tiktoken

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt", encoding="utf-8").read()
enc = tiktoken.get_encoding("gpt2")
ids = enc.encode(text)
split = int(0.9 * len(ids))
train, val = ids[:split], ids[split:]
V = enc.n_vocab
print(len(text), len(ids), f"{len(text.encode()) / len(ids):.2f} bytes per token")

pair = Counter(zip(train[:-1], train[1:]))             # c(a, b)
first = Counter(train[:-1])                            # c(a)

def bigram_loss(tokens, alpha):
    """Average -log p(b | a) with add-alpha smoothing over all V tokens."""
    nll = 0.0
    for a, b in zip(tokens[:-1], tokens[1:]):
        nll -= math.log((pair[(a, b)] + alpha) / (first[a] + alpha * V))
    return nll / (len(tokens) - 1)

print(f"uniform model: {math.log(V):.3f}")
for alpha in [0.1, 0.01, 0.001, 0.0001]:
    tr, va = bigram_loss(train, alpha), bigram_loss(val, alpha)
    print(f"alpha={alpha}: train {tr:.3f}, val {va:.3f} (perplexity {math.exp(va):.0f})")
# 1115394 338025 3.30 bytes per token
# uniform model: 10.825
# alpha=0.1: train 6.057, val 6.878 (perplexity 970)
# alpha=0.01: train 4.672, val 6.095 (perplexity 444)
# alpha=0.001: train 3.849, val 5.859 (perplexity 350)
# alpha=0.0001: train 3.501, val 6.074 (perplexity 435)
